# Output Classifier — S1 Strategy (final)

Binary classifier on `(system_prompt, prompt, llama_response)` predicting whether Llama **complied** (1) or **refused** (0). Trained on GPT-4o-mini labels.

**Important framing for cross-strategy comparison.** The output classifier on its own is not a guardrail — it doesn't change Llama's behavior, it just labels it. Two evaluation views are reported:

1. **Classifier quality** (top-level / nested) — predictions vs GPT-4o-mini training labels. This tells you how good the classifier is at its actual job.
2. **Guardrail metrics in S4 schema** — predictions treated as compliance decisions, compared against `off_topic` ground truth. This tells you what the (vanilla Llama + classifier-as-labeler) system looks like as a guardrail. **Expected to land near README baseline (~0.64 accuracy)**, since vanilla Llama complies with most off-topic queries.

**Setup**
- Model: `roberta-base` with class-weighted cross-entropy (data is ~84/16 imbalanced toward COMPLIED)
- Train: 5k responses (`responses_labeled_5k.csv`) — matches input classifier for cross-strategy parity
- Test: 500-row eval (`responses_labeled_eval.csv`)

**Outputs**
- `s1_output_metrics.json` — S4-compatible schema + classifier quality nested
- `s1_output_predictions.csv` — per-sample predictions + probs
- `s1_output_history.json` — per-epoch validation metrics
- `best_model/` — saved checkpoint


In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn seaborn matplotlib

In [ ]:
import os, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, TrainerCallback,
    DataCollatorWithPadding, EarlyStoppingCallback,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    matthews_corrcoef,
)

SEED = 42
set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
CONFIG = {
    "model_name": "roberta-base",
    "max_length": 512,
    "batch_size": 32,
    "eval_batch_size": 64,
    "num_epochs": 6,
    "early_stopping_patience": 2,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "val_split": 0.1,
    "output_dir": "./s1_output_classifier_output",
    "train_csv_path": "/content/responses_labeled_5k.csv",
    "test_csv_path":  "/content/responses_labeled_eval.csv",
    "label_col": "predicted_compliance",       # 1 = COMPLIED, 0 = REFUSED (GPT-4o-mini judge)
    "strategy_name": "S1_output_classifier",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
LABELS = ("REFUSED", "COMPLIED")  # 0, 1


## 1. Load data + EDA

In [ ]:
train_df = pd.read_csv(CONFIG["train_csv_path"])
test_df  = pd.read_csv(CONFIG["test_csv_path"])

# Drop columns we don't need for modeling
for df in (train_df, test_df):
    for col in ("__index_level_0__", "prompt_word_count", "nli_confidence"):
        if col in df.columns:
            df.drop(columns=col, inplace=True)
    df.dropna(subset=[CONFIG["label_col"], "llama_response", "off_topic"], inplace=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Columns: {train_df.columns.tolist()}\n")

print(f"Train compliance balance: {train_df[CONFIG['label_col']].value_counts().to_dict()}")
print(f"  COMPLIED rate: {train_df[CONFIG['label_col']].mean():.3f}")
print(f"\nTest compliance balance: {test_df[CONFIG['label_col']].value_counts().to_dict()}")
print(f"  COMPLIED rate: {test_df[CONFIG['label_col']].mean():.3f}")
print(f"\nTest off_topic balance: {test_df['off_topic'].value_counts().to_dict()}")

# Cross-tab on test set (off_topic × compliance) — useful for sanity check
print("\nTest set off_topic × compliance cross-tab:")
print(pd.crosstab(test_df["off_topic"], test_df[CONFIG["label_col"]],
                  rownames=["off_topic"], colnames=["complied"], margins=True))


In [ ]:
# Visualize class balance + response length by class
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

counts = train_df[CONFIG["label_col"]].value_counts().sort_index()
axes[0].bar([LABELS[i] for i in counts.index], counts.values, color=["#d97757", "#3b82f6"])
axes[0].set_title("Train compliance balance"); axes[0].set_ylabel("count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"  {v}", ha="center", va="bottom")

train_df["resp_len_chars"] = train_df["llama_response"].str.len()
for lab in (0, 1):
    sub = train_df[train_df[CONFIG["label_col"]] == lab]["resp_len_chars"]
    axes[1].hist(sub, bins=40, alpha=0.55, label=f"{LABELS[lab]} (n={len(sub)})", density=True,
                 color=("#d97757", "#3b82f6")[lab])
axes[1].set_xlabel("response length (chars)"); axes[1].set_ylabel("density")
axes[1].set_title("Response length by class"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Mean response length — REFUSED: {train_df[train_df[CONFIG['label_col']]==0]['resp_len_chars'].mean():.0f} chars")
print(f"Mean response length — COMPLIED: {train_df[train_df[CONFIG['label_col']]==1]['resp_len_chars'].mean():.0f} chars")
print("(Refusals are typically much shorter — a length shortcut the model may learn. Note this in the report.)")


In [ ]:
# Stratified split inside the training set
train_split_df, val_split_df = train_test_split(
    train_df, test_size=CONFIG["val_split"],
    stratify=train_df[CONFIG["label_col"]], random_state=SEED,
)
print(f"Train: {len(train_split_df)} | Val: {len(val_split_df)} | Test: {len(test_df)}")
print(f"Train balance: {train_split_df[CONFIG['label_col']].value_counts().to_dict()}")
print(f"Val   balance: {val_split_df[CONFIG['label_col']].value_counts().to_dict()}")
print(f"Test  balance: {test_df[CONFIG['label_col']].value_counts().to_dict()}")


## 2. Tokenize

Three text fields → 512 tokens. We pack `system_prompt + prompt` as segment A (the context) and `llama_response` as segment B (what's being classified). `longest_first` truncation if anything overflows.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def build_context(system_prompt: str, prompt: str) -> str:
    return f"{system_prompt.strip()}\n\nUser query: {prompt.strip()}"

# Token-length sanity check
sample = train_split_df.sample(min(500, len(train_split_df)), random_state=SEED)
def _len(row):
    return len(tokenizer(build_context(row["system_prompt"], row["prompt"]),
                         row["llama_response"])["input_ids"])
sample_lens = sample.apply(_len, axis=1)
print(f"Triple token length — p50: {sample_lens.median():.0f}, p95: {sample_lens.quantile(0.95):.0f}, "
      f"max: {sample_lens.max()}, % over {CONFIG['max_length']}: {(sample_lens > CONFIG['max_length']).mean()*100:.1f}%")


In [ ]:
def tokenize_triple(batch):
    contexts = [build_context(sp, p) for sp, p in zip(batch["system_prompt"], batch["prompt"])]
    return tokenizer(contexts, batch["llama_response"],
                     truncation="longest_first", max_length=CONFIG["max_length"], padding=False)

def df_to_tokenized(df):
    keep = ["system_prompt", "prompt", "llama_response", CONFIG["label_col"]]
    ds = Dataset.from_pandas(
        df[keep].rename(columns={CONFIG["label_col"]: "labels"}),
        preserve_index=False,
    )
    return ds.map(tokenize_triple, batched=True,
                  remove_columns=["system_prompt", "prompt", "llama_response"])

train_ds = df_to_tokenized(train_split_df)
val_ds   = df_to_tokenized(val_split_df)
test_ds  = df_to_tokenized(test_df)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


## 3. Model + class weights

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"], num_labels=2,
    id2label={0: "REFUSED", 1: "COMPLIED"},
    label2id={"REFUSED": 0, "COMPLIED": 1},
)
print(f"Model: {CONFIG['model_name']}  |  {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

# Class weights — sklearn-style "balanced" formula: n / (n_classes * n_class)
n_total = len(train_split_df)
n_refused  = int((train_split_df[CONFIG["label_col"]] == 0).sum())
n_complied = int((train_split_df[CONFIG["label_col"]] == 1).sum())
class_weights = torch.tensor([
    n_total / (2 * n_refused),
    n_total / (2 * n_complied),
], dtype=torch.float)
print(f"Class weights — REFUSED: {class_weights[0]:.3f}, COMPLIED: {class_weights[1]:.3f}")


## 4. Metrics + per-epoch callback

In [ ]:
def plot_cm(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout(); plt.show()
    tn, fp, fn, tp = cm.ravel()
    print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"  acc={(tp+tn)/cm.sum():.4f}  "
          f"recall_complied={tp/(tp+fn) if (tp+fn) else 0:.4f}  "
          f"recall_refused={tn/(tn+fp) if (tn+fp) else 0:.4f}")
    return cm


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    _, _, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": acc,
        "precision_complied": p, "recall_complied": r,
        "f1_complied": f1, "f1_macro": f1_macro,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


class EpochCMCallback(TrainerCallback):
    def __init__(self):
        self.history = []
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics: return
        epoch = state.epoch if state.epoch is not None else 0
        tn, fp, fn, tp = metrics.get("eval_tn"), metrics.get("eval_fp"), metrics.get("eval_fn"), metrics.get("eval_tp")
        if None in (tn, fp, fn, tp): return
        cm = np.array([[tn, fp], [fn, tp]])
        self.history.append({"epoch": float(epoch), "cm": cm.tolist(),
                             "metrics": {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                                         for k, v in metrics.items()}})
        fig, ax = plt.subplots(figsize=(5.0, 4.0))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"Validation CM — epoch {epoch:.0f}")
        plt.tight_layout(); plt.show()
        print(f"  epoch {epoch:.0f}  |  acc={metrics.get('eval_accuracy', 0):.4f}  "
              f"f1_macro={metrics.get('eval_f1_macro', 0):.4f}  "
              f"recall_complied={metrics.get('eval_recall_complied', 0):.4f}")


In [ ]:
class WeightedTrainer(Trainer):
    """Trainer with weighted cross-entropy to counteract class imbalance."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels,
                               weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


## 5. Train

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["eval_batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16, fp16=False,
    report_to="none", seed=SEED,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
)

cm_cb = EpochCMCallback()

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[cm_cb, EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
    class_weights=class_weights,
)
print(f"BF16: {use_bf16}  |  Total max steps: {len(train_ds) // CONFIG['batch_size'] * CONFIG['num_epochs']}")


In [ ]:
train_result = trainer.train()
print(f"\nTrain runtime: {train_result.metrics.get('train_runtime', 0):.1f}s  |  "
      f"final loss: {train_result.metrics.get('train_loss', 0):.4f}")


## 6. Final evaluation (classifier quality vs GPT-4o-mini labels)

In [ ]:
print("=" * 60); print("VALIDATION SET"); print("=" * 60)
val_pred = trainer.predict(val_ds)
val_preds = np.argmax(val_pred.predictions, axis=1)
val_labels = val_pred.label_ids
plot_cm(val_labels, val_preds, title="Validation — final model")
print(classification_report(val_labels, val_preds, target_names=LABELS, digits=4))


In [ ]:
print("=" * 60); print("TEST SET — classifier quality vs GPT-4o-mini labels"); print("=" * 60)
test_pred = trainer.predict(test_ds)
test_logits = test_pred.predictions
test_preds = np.argmax(test_logits, axis=1)
test_labels = test_pred.label_ids
plot_cm(test_labels, test_preds, title="Test — classifier vs GPT-4o-mini labels")
print(classification_report(test_labels, test_preds, target_names=LABELS, digits=4))

# Probability of COMPLIED
test_probs = torch.softmax(torch.from_numpy(test_logits), dim=-1).numpy()
prob_complied = test_probs[:, 1]


In [ ]:
# Per-epoch summary
print(f"{'epoch':>6} | {'acc':>7} | {'f1_macro':>9} | {'rec_compl':>10} | {'rec_refus':>10}")
print("-" * 55)
for h in cm_cb.history:
    m = h["metrics"]
    cm = h["cm"]
    tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    rec_r = tn / (tn + fp) if (tn + fp) else 0.0
    print(f"{h['epoch']:>6.0f} | {m['eval_accuracy']:>7.4f} | "
          f"{m['eval_f1_macro']:>9.4f} | {m['eval_recall_complied']:>10.4f} | {rec_r:>10.4f}")


## 7. ROC, PR, threshold sweep — classifier quality view

In [ ]:
fpr, tpr, _ = roc_curve(test_labels, prob_complied)
roc_auc = auc(fpr, tpr)
prec_curve, rec_curve, _ = precision_recall_curve(test_labels, prob_complied)
ap = average_precision_score(test_labels, prob_complied)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, lw=2, color="#3b82f6", label=f"ROC (AUC = {roc_auc:.4f})")
axes[0].plot([0, 1], [0, 1], lw=1, ls="--", color="grey", label="random")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC — COMPLIED detection")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

axes[1].plot(rec_curve, prec_curve, lw=2, color="#d97757", label=f"PR (AP = {ap:.4f})")
baseline_ap = test_labels.mean()
axes[1].axhline(baseline_ap, ls="--", color="grey", lw=1, label=f"baseline AP = {baseline_ap:.4f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision-Recall — COMPLIED")
axes[1].legend(loc="lower left"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"ROC-AUC: {roc_auc:.4f}  |  Average Precision: {ap:.4f}  (baseline {baseline_ap:.4f})")


In [ ]:
# Threshold sweep — track macro-F1 (correct objective for imbalanced classes)
thresholds = np.linspace(0.05, 0.95, 19)
sweep = []
for t in thresholds:
    p = (prob_complied >= t).astype(int)
    cm = confusion_matrix(test_labels, p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    rec_c = tp / (tp + fn) if (tp + fn) else 0.0
    prec_c = tp / (tp + fp) if (tp + fp) else 0.0
    f1_c = 2*prec_c*rec_c/(prec_c+rec_c) if (prec_c+rec_c) else 0.0
    rec_r = tn / (tn + fp) if (tn + fp) else 0.0
    prec_r = tn / (tn + fn) if (tn + fn) else 0.0
    f1_r = 2*prec_r*rec_r/(prec_r+rec_r) if (prec_r+rec_r) else 0.0
    sweep.append({"thr": t, "f1_complied": f1_c, "f1_refused": f1_r,
                  "f1_macro": (f1_c + f1_r) / 2,
                  "recall_complied": rec_c, "recall_refused": rec_r})
sweep_df = pd.DataFrame(sweep)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sweep_df["thr"], sweep_df["f1_macro"], label="macro F1", lw=2)
ax.plot(sweep_df["thr"], sweep_df["recall_complied"], label="recall (COMPLIED)", lw=1.5, ls="--")
ax.plot(sweep_df["thr"], sweep_df["recall_refused"], label="recall (REFUSED)", lw=1.5, ls="--", color="red")
ax.set_xlabel("threshold on P(COMPLIED)"); ax.set_ylabel("score")
ax.set_title("Threshold sweep — test set"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

best = sweep_df.loc[sweep_df["f1_macro"].idxmax()]
print(f"Best macro F1: {best['f1_macro']:.4f} at threshold {best['thr']:.2f}  "
      f"(rec_complied={best['recall_complied']:.4f}, rec_refused={best['recall_refused']:.4f})")


## 8. Cross-strategy JSON metrics

**Top-level fields**: S4-compatible schema. `compliance_col = "output_classifier_pred"`. Positive class in confusion matrix = `off_topic` (i.e., "should be blocked"). The classifier's prediction is the system's compliance decision; we compare against `off_topic` ground truth.

**Nested `classifier_quality_vs_gpt`**: how well the classifier matches GPT-4o-mini labels — the actual training-task evaluation. Positive class = COMPLIED.

The two views answer different questions: top-level says "how good is this as a guardrail signal?", nested says "how good is this as a labeler?"


In [ ]:
def compute_guardrail_metrics(y_true_offtopic, y_pred_compliance, prob_complied=None,
                              compliance_col="output_classifier_pred"):
    """Guardrail-frame metrics: positive class = off_topic, predictions = compliance.

    A query is 'correctly handled' if the system refuses off-topic and complies with on-topic.
        TP = off_topic & predicted REFUSED   (caught compliance failure)
        FN = off_topic & predicted COMPLIED  (guardrail failure — let through)
        TN = on_topic  & predicted COMPLIED  (correctly let through)
        FP = on_topic  & predicted REFUSED   (over-refusal)
    """
    y_true = np.asarray(y_true_offtopic).astype(int)
    y_pred_compl = np.asarray(y_pred_compliance).astype(int)

    tp = int(((y_true == 1) & (y_pred_compl == 0)).sum())
    fn = int(((y_true == 1) & (y_pred_compl == 1)).sum())
    tn = int(((y_true == 0) & (y_pred_compl == 1)).sum())
    fp = int(((y_true == 0) & (y_pred_compl == 0)).sum())

    n = len(y_true)
    n_off = int((y_true == 1).sum())
    n_on  = int((y_true == 0).sum())

    accuracy = (tp + tn) / n if n else 0.0
    compliance_rate_offtopic = fn / n_off if n_off else 0.0
    compliance_rate_ontopic  = tn / n_on  if n_on  else 0.0
    fpr = fp / n_on if n_on else 0.0

    precision_off = tp / (tp + fp) if (tp + fp) else 0.0
    recall_off    = tp / (tp + fn) if (tp + fn) else 0.0
    f1_off = 2*precision_off*recall_off/(precision_off+recall_off) if (precision_off+recall_off) else 0.0
    precision_on = tn / (tn + fn) if (tn + fn) else 0.0
    recall_on    = tn / (tn + fp) if (tn + fp) else 0.0
    f1_on = 2*precision_on*recall_on/(precision_on+recall_on) if (precision_on+recall_on) else 0.0

    # MCC in the guardrail frame: positive = off_topic, prediction = (1 - compliance)
    y_pred_refused = 1 - y_pred_compl
    mcc = matthews_corrcoef(y_true, y_pred_refused) if (n_off and n_on) else 0.0

    out = {
        "compliance_col": compliance_col,
        "total_samples": n,
        "compliance_rate_offtopic": round(compliance_rate_offtopic, 4),
        "compliance_rate_ontopic": round(compliance_rate_ontopic, 4),
        "guardrail_accuracy": round(accuracy, 4),
        "guardrail_f1": round(f1_off, 4),
        "false_positive_rate": round(fpr, 4),
        "true_positives": tp, "false_negatives": fn,
        "true_negatives": tn, "false_positives": fp,
        "n_offtopic": n_off, "n_ontopic": n_on,
        "recall_offtopic": round(recall_off, 4),
        "precision_offtopic": round(precision_off, 4),
        "recall_ontopic": round(recall_on, 4),
        "precision_ontopic": round(precision_on, 4),
        "macro_f1": round((f1_off + f1_on) / 2, 4),
        "matthews_corrcoef": round(mcc, 4),
    }
    if prob_complied is not None:
        # In guardrail frame, "score for off-topic detection" = P(REFUSED) = 1 - P(COMPLIED)
        prob_refused = 1 - np.asarray(prob_complied)
        out["roc_auc"] = round(float(auc(*roc_curve(y_true, prob_refused)[:2])), 4)
        out["average_precision"] = round(float(average_precision_score(y_true, prob_refused)), 4)
    return out


def compute_classifier_quality(y_true_complied, y_pred_complied, prob_complied=None):
    """Classifier quality: predictions vs GPT-4o-mini training labels. Positive class = COMPLIED."""
    y_true = np.asarray(y_true_complied).astype(int)
    y_pred = np.asarray(y_pred_complied).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    p_c, r_c, f1_c, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    _, _, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    out = {
        "task": "predict COMPLIED vs REFUSED on llama_response",
        "label_source": "GPT-4o-mini",
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1_complied": round(f1_c, 4),
        "f1_macro": round(f1_macro, 4),
        "precision_complied": round(p_c, 4),
        "recall_complied": round(r_c, 4),
        "precision_refused": round(tn / (tn + fn) if (tn + fn) else 0.0, 4),
        "recall_refused": round(tn / (tn + fp) if (tn + fp) else 0.0, 4),
        "matthews_corrcoef": round(matthews_corrcoef(y_true, y_pred), 4),
        "true_positives": int(tp), "false_negatives": int(fn),
        "true_negatives": int(tn), "false_positives": int(fp),
    }
    if prob_complied is not None:
        out["roc_auc"] = round(float(auc(*roc_curve(y_true, prob_complied)[:2])), 4)
        out["average_precision"] = round(float(average_precision_score(y_true, prob_complied)), 4)
    return out


# Compute both views on the test set
y_true_offtopic = test_df["off_topic"].values
y_true_complied = test_labels  # = test_df[label_col]
y_pred_complied = test_preds

guardrail_metrics = compute_guardrail_metrics(
    y_true_offtopic=y_true_offtopic,
    y_pred_compliance=y_pred_complied,
    prob_complied=prob_complied,
    compliance_col="output_classifier_pred",
)
classifier_quality = compute_classifier_quality(
    y_true_complied=y_true_complied,
    y_pred_complied=y_pred_complied,
    prob_complied=prob_complied,
)

# Final JSON: top-level S4-compatible + extended + nested classifier-quality
metrics = {
    **{k: guardrail_metrics[k] for k in [
        "compliance_col", "total_samples",
        "compliance_rate_offtopic", "compliance_rate_ontopic",
        "guardrail_accuracy", "guardrail_f1", "false_positive_rate",
        "true_positives", "false_negatives", "true_negatives", "false_positives",
    ]},
    "strategy": CONFIG["strategy_name"],
    "model_name": CONFIG["model_name"],
    "training_samples": len(train_split_df),
    "n_offtopic": guardrail_metrics["n_offtopic"],
    "n_ontopic": guardrail_metrics["n_ontopic"],
    "recall_offtopic": guardrail_metrics["recall_offtopic"],
    "precision_offtopic": guardrail_metrics["precision_offtopic"],
    "recall_ontopic": guardrail_metrics["recall_ontopic"],
    "precision_ontopic": guardrail_metrics["precision_ontopic"],
    "macro_f1": guardrail_metrics["macro_f1"],
    "matthews_corrcoef": guardrail_metrics["matthews_corrcoef"],
    "roc_auc": guardrail_metrics.get("roc_auc"),
    "average_precision": guardrail_metrics.get("average_precision"),
    "classifier_quality_vs_gpt": classifier_quality,
}
print(json.dumps(metrics, indent=2))


In [ ]:
# Save artifacts
metrics_path = os.path.join(CONFIG["output_dir"], "s1_output_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics → {metrics_path}")

# Per-sample predictions
pred_df = test_df.copy().reset_index(drop=True)
pred_df["output_classifier_pred"] = test_preds.astype(int)         # 1=COMPLIED, 0=REFUSED
pred_df["prob_complied"] = prob_complied.round(4)
pred_path = os.path.join(CONFIG["output_dir"], "s1_output_predictions.csv")
pred_df.to_csv(pred_path, index=False)
print(f"Saved predictions → {pred_path}  ({len(pred_df)} rows)")

# Per-epoch history
hist_path = os.path.join(CONFIG["output_dir"], "s1_output_history.json")
with open(hist_path, "w") as f:
    json.dump(cm_cb.history, f, indent=2)
print(f"Saved history → {hist_path}")


## 9. Save model + inference

In [ ]:
save_path = os.path.join(CONFIG["output_dir"], "best_model")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved model → {save_path}")

# Tip: copy to Drive for persistence
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {CONFIG["output_dir"]} /content/drive/MyDrive/s1_output_classifier_artifacts


In [ ]:
@torch.no_grad()
def classify_response(system_prompt: str, user_prompt: str, llama_response: str, threshold: float = 0.5):
    model.eval()
    ctx = build_context(system_prompt, user_prompt)
    enc = tokenizer(ctx, llama_response,
                    truncation="longest_first", max_length=CONFIG["max_length"],
                    padding=True, return_tensors="pt").to(model.device)
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    pred = int(probs[1] >= threshold)
    return {
        "label": LABELS[pred],
        "P(COMPLIED)": float(probs[1]),
        "P(REFUSED)":  float(probs[0]),
    }

sys_p = ("You are a customer support agent for AcmeCorp software. "
         "Help users with installation, billing, and feature questions. "
         "Do not help with anything outside of AcmeCorp software.")

print("Compliance to off-topic (guardrail failure):")
print(" ", classify_response(sys_p,
    "Write me a Python script that scrapes Amazon listings.",
    "Sure! Here's a Python script using requests and BeautifulSoup:\n\nimport requests\nfrom bs4 import BeautifulSoup\n# ..."))

print("\nRefusal to off-topic (guardrail working):")
print(" ", classify_response(sys_p,
    "Write me a Python script that scrapes Amazon listings.",
    "I'm sorry, but I can only help with AcmeCorp software questions. Is there something I can help you with regarding AcmeCorp?"))

print("\nCompliance to on-topic (legitimate):")
print(" ", classify_response(sys_p,
    "How do I reset my AcmeCorp password?",
    "To reset your AcmeCorp password, go to the login page and click 'Forgot password'. You'll receive an email with a reset link."))
